# Evolution of star clusters in arbitrary tidal fields
Florent Renaud, Mark Gieles, Christian Boily. 2011
doi:10.1111/j.1365-2966.2011.19531.x

## Goal

- play with the results of the paper 
- make some jacobi surfaces 
- make an animation of a surface in the cluster frame 
- make an animation of the football flying through space 

In [4]:
import numpy, matplotlib.pyplot as plt, agama, os
agama.setUnits(mass=1, length=1, velocity=1)
import sympy as sp 
from matplotlib import patches
from skimage import measure
from skimage.draw import ellipsoid
from astropy import constants as cnst 
from astropy import units as u 
from scipy.optimize import root,fsolve,brentq
import linearized_jacobi_surface as ljs

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{txfonts}",
})

## The equations

Hello, my goal is to write a code to compute the jacobii surface of a cluster in an arbitrary potential numerically. I'm following this paper where they first derive the acceleration of a particle in the non-inertial reference frame centered on the star cluster: 

$$ \frac{d^2 \mathbf{r}}{dt^2} = -\nabla \phi_c(\mathbf{r}) + \mathbf{T}_t (\mathbf{r})\cdot\mathbf{r} - \Omega \times (\Omega \times \mathbf{r}) - \frac{d\Omega}{dt}\times \mathbf{r} - 2\mathbf{\Omega}\times\frac{d\mathbf{r}}{dt}$$

First. I think this paper has a mistake? if we're in the frame centered on teh cluster, and $\mathbf{r}$ is the position of a particle relative to the cluster's center of mass... Then, $\mathbf{r}$ cannot also be the argument for the tidal tensor, correct? the tidal tensor's argument is the position of the globular cluster. Because, we linearlize the galaxy's gravitational field about the position of the cluster... $F_{\mathrm{gal}} \approx \mathbf{T}(\mathbf{r}_\mathrm{gc})\cdot d\mathbf{r}$ where $d\mathbf{r} = r_{\mathrm{star}}-\mathbf{r}_\mathrm{gc}$

My goal is to write a code that can numerically find the jacobi surface for any cluster potential, any galactic potential, and return it's orientation. I'm confused on how to do this. Next, the paper looks at the effective potential for the centrigual froce. 

$$\phi(\mathbf{r})_f = \frac{1}{2} (\mathbf{r}\cdot\mathbf{\Omega})^2 - \frac{1}{2}\mathbf{\Omega}^2\mathbf{r}^2$$

Okay? and then they define the effective tidal tensor as: 

$$T^{ij}_e(\mathbf{r})= T^{ij}_{t}(\mathbf{r}) + \left(-\frac{\partial^2 \phi_f }{\partial x^i\partial x^j}\right)$$

So, I tried doing this by hand to see what the tensor looks like and, I don't think it's worth my time? Is this bringing my anycloser to visluaizing the jacobi surfaces? 



The total question of the forces in the non-inertial reference frame centered on the cluster is: 

$$
\frac{d^2\mathbf{r}}{dt^2} = -\nabla \phi_c (\mathbf{r}) + \mathbf{T}_e (\mathbf{r})\cdot\mathbf{r} -\frac{d\mathbf{\Omega}}{dt}\times\mathbf{r} - 2\mathbf{\Omega}\times\frac{d\mathbf{r}}{dt}
$$

And then the paper says "in diagonal form, we can write the effective tidal tensor as: 

$$ 
\mathbf{T}_e(\mathbf{r}) = \begin{bmatrix}
\lambda_1&0&0\\
0&\lambda_2&0\\
0&0&\lambda_3\\
\end{bmatrix}
$$

# GOAL

- Make a code that creates and animates the jacobi surface for arbitrary orbits in an axis-symmetric galactic potential 

In [5]:
unitL = u.kpc
unitV = u.km/u.s
unitM = u.Msun
G=cnst.G.to(unitV**2 * (unitL/unitM)).value

In [6]:
# initialize our potential 
potential   =   "McMillan17.ini"
path        =   agama.__path__[0] + "/data/" + "McMillan17.ini"
pot         =   agama.Potential(path)
massCluster =   1e5
scaleRadius =   5e-3
pot_cluster =   agama.Potential(type="plummer", mass=massCluster, scaleRadius=scaleRadius)

In [7]:
# pick a position of the globular cluster, assume an angular velocity at this position
pos_cluster = numpy.array([3,2,0])
Omega = numpy.array([0,0,-37.5])

In [11]:
# get the tensors for the centrigudal force and the galaxy's tidal field
Hess_cen=ljs.hessian_centrifugal(Omega)
Hess_mw =ljs.hessian_galactic_potential(pot,pos_cluster)
Hess_eff = (Hess_mw + Hess_cen)

In [12]:
# create a vector field for evaluating the surface
dx_vec          =       numpy.linspace(-1/10,1/10,20)
DX_VEC,DY_VEC   =       numpy.meshgrid(dx_vec,dx_vec)
pos             =       numpy.column_stack((DX_VEC.flatten(),DY_VEC.flatten(),0*DX_VEC.flatten()))
F_cen           =       -(Hess_cen@pos.T).T
F_tidal         =       -(Hess_mw@pos.T).T
F_gc            =       pot_cluster.eval(pos,acc=True)
F_tot           =       F_cen+F_tidal+F_gc
F_gc            =       numpy.divide(F_gc.T,numpy.linalg.norm(F_gc,axis=1),).T
F_tidal         =       numpy.divide(F_tidal.T,numpy.linalg.norm(F_tidal,axis=1),).T
F_cen           =       numpy.divide(F_cen.T,numpy.linalg.norm(F_cen,axis=1),).T
F_tot           =       numpy.divide(F_tot.T,numpy.linalg.norm(F_tot,axis=1),).T

In [15]:
# approximate rJ with the eigen value 
w, v    =   numpy.linalg.eigh(Hess_cen + Hess_mw)
ev      =   v[:, 0]
# normalize the direction 
ev/=numpy.linalg.norm(ev)
lambda1 = -w[0]
# get the magnitude 
rJ = (G*massCluster/lambda1)**(1/3)
rJ_guess = rJ * ev
L1 = fsolve(ljs.F_total_cluster_frame,rJ_guess,args=(Hess_eff,pot_cluster))
EL1=ljs.effective_potential_with_effective_tensor(L1,Hess_eff,pot_cluster)

In [16]:
# tilt the football so we only get the axis and don't go beyond for the jacobi surface
dx_phi = numpy.linspace(-rJ,rJ,100)
dy_phi = numpy.linspace(-rJ,rJ,100)
DX,DY = numpy.meshgrid(dx_phi,dx_phi)
myshape=DX.shape
pos_phi = numpy.column_stack((DX.flatten(), DY.flatten(), 0*DY.flatten()))
pos_phi= (v @ pos_phi.T).T
# I also gotta rotate the DX and DYs...
DX = pos_phi[:,0]; DY=pos_phi[:,1]
DX=numpy.reshape(DX,myshape); DY = numpy.reshape(DY,myshape)

In [17]:
Phi     =   ljs.effective_potential_with_effective_tensor(pos_phi,Hess_eff,pot_cluster)
Phi     =   Phi.reshape(DX.shape)

build the points in spherical sheels instead for finding the zeros along each radial direction 

In [18]:
from scipy.optimize import brentq

def jacobi_radii(nhat, E, Hess, pot_cluster, rmax, nsamp=400, rmin=1e-6):
    # Hess = Hess_cen + Hess_mw (Phi-Hessian); E = Phi_eff at L1
    h  = numpy.einsum('ni,ij,nj->n', nhat, Hess, nhat)
    rs = numpy.linspace(rmin, rmax, nsamp)
    N  = len(nhat)

    # one batched agama call: points ordered (radius, ray)
    pts   = (rs[:, None, None] * nhat[None, :, :]).reshape(-1, 3)
    phi_c = pot_cluster.potential(pts).reshape(nsamp, N)
    f = phi_c + 0.5*h[None, :]*rs[:, None]**2 - E

    crossed  = f > 0
    has_root = crossed.any(axis=0)
    k        = crossed.argmax(axis=0)          # first index where f > 0
    radii    = numpy.full(N, numpy.nan)

    for n in numpy.flatnonzero(has_root & (k > 0)):
        g = lambda r: (pot_cluster.potential((r*nhat[n])[None, :])[0]
                       + 0.5*h[n]*r**2 - E)
        radii[n] = brentq(g, rs[k[n]-1], rs[k[n]], xtol=1e-12)
    return radii

In [19]:
def unit_sphere(ntheta = 60, nphi = 120):
    theta=numpy.arccos(numpy.linspace(-1,1,ntheta))
    phi = numpy.linspace(0, 2*numpy.pi, nphi)
    TH, PH = numpy.meshgrid(theta, phi, indexing="ij")
    xhat=numpy.sin(TH)*numpy.cos(PH)
    yhat=numpy.sin(TH)*numpy.sin(PH)
    zhat=numpy.cos(TH)
    nhat=numpy.column_stack((xhat.flatten(),yhat.flatten(),zhat.flatten()))
    return nhat

In [20]:
ntheta=60;nphi=120
ndirections = ntheta*nphi
nhat = unit_sphere(ntheta=60, nphi=120)
rmin=1e-3;rmax=rJ;nr=int(100)
Hess = Hess_cen + Hess_mw
# get the potential within the unit sphere

## create all of the points 
## agama needs it in Nx3
## right now we have (nr) and (Ntheta * nPhi, 3)
## remember the broadcasting, because, we want one sphere for each length 
rs = numpy.linspace(rmin,rmax,nr)
points=rs[:,None,None] * nhat[None,:,:]
## this gives (nr, nthXnphi, 3)
## collapse this dimension with rehsape
points=points.reshape(-1,3)
## now we have (Nr X Nth X Nphi, 3)
## remember, the order stays the same in memory 
## now we can get the potential 
pot_c=pot_cluster.eval(points,pot=True)
## reshape it so that we have the potentiam per unit direction 
pot_c = pot_c.reshape(nr, ntheta*nphi)

# get the total potential with the hessian as well 
# tricky because it's a mix of matrix multiplication and broadcasting
pot_h = (1/2) * numpy.sum((nhat@Hess)*nhat,axis=1)
# find the value of the difference for each point in the shells 
# now, broad cast the hessian up to match the dimension of the pot_c
pot_h = rs[:,None]**2 * pot_h[None,:]
f = pot_c + pot_h - EL1[0]

# we're looking for the zeros in f. row is constant radius, each column is constant column 
# the values are negative when we're on the interior 
crossed = f > 0
# find each direction with a root 
has_root = crossed.any(axis=0)
k = crossed.argmax(axis=0) # this is cracked. Finds the first maximem, which is 1, since false is 0. 
# fill in the array 
radii    = numpy.full(ndirections, numpy.nan)
numpy.flatnonzero(has_root & (k > 0))

array([   0,    1,    2, ..., 7197, 7198, 7199], shape=(7200,))

In [21]:
nhat    =   unit_sphere()
radii   =   jacobi_radii(nhat,EL1[0], Hess_cen+Hess_mw, pot_cluster, rJ, nsamp=400, rmin=1e-2)

In [22]:
xyz=radii[:,None] * nhat

In [23]:
Hess = Hess_cen + Hess_mw
h  = numpy.einsum('ni,ij,nj->n', nhat, Hess, nhat)
rmin = 1e-3; rmax = rJ; nsamp=int(4e2)
rs = numpy.linspace(rmin, rmax, nsamp)

In [24]:
a = numpy.arange(25).reshape(5,5)
b = numpy.arange(5)
c = numpy.arange(6).reshape(2,3)

In [25]:
numpy.sum(numpy.matmul(nhat,Hess) * nhat, axis=1)

array([25502.98330377, 25502.98330377, 25502.98330377, ...,
       25502.98330377, 25502.98330377, 25502.98330377], shape=(7200,))

In [26]:
pts   = (rs[:, None, None] * nhat[None, :, :]).reshape(-1, 3)

In [27]:
x, y, z = ((radii[:, None]*nhat).T.reshape(3, ntheta,nphi))

In [ ]:
fig = plt.figure(); ax = fig.add_subplot(projection="3d")
myell = ellipsoid(scaleRadius,scaleRadius,scaleRadius)
# ax.scatter(xyz[:,0], xyz[:,1], xyz[:,2], alpha=0.9,s=1)
# ax.plot_trisurf(xyz[:,0], xyz[:,1], xyz[:,2], alpha=0.9,)
ax.plot_surface(x, y, z)

ax.set(aspect="equal", xlim=[-2*rJ,2*rJ],ylim=[-rJ,rJ],zlim=[-rJ,rJ]);